# Notebook11 evaluation reusable definitions

Purpose: source-bound overnight full-JEPA comparison of ViT, spatial kernel and learned geometry kernel encoders. Inputs: registered configuration, six fresh paired seeds and audited Food101 training/development membership. Outputs: immutable initial states, interruption-safe full-state checkpoints and all54initial/350/final fixedepoch20 probes. Import only `ahr-export` cells; imports never train. Expected runtime: definition import seconds; consumer study has an original twelve-hour cap. Checkpoints: `checkpoints/<run_id>/`; retain latest two valid resumable states and best. Resume: Run All in `notebooks/11_Overnight_Manifold_JEPA.ipynb`, cell `launch-overnight-manifold`, with the same run ID and unchanged sources before the original deadline. One writer per run; invalid evidence refuses restart. Final update budget is selected from discarded timing/memory calibration before quality outcomes.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import nbformat
ROOT=Path(os.environ.get("FMM_JEPA_ROOT","/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
_boot={"__file__":str(ROOT/"fmm_jepa/bootstrap.ipynb")}
for _cell in nbformat.read(ROOT/"fmm_jepa/bootstrap.ipynb",as_version=4).cells:
    if _cell.cell_type=="code" and "ahr-export" in _cell.metadata.get("tags",[]):
        exec(compile(_cell.source,"bootstrap#ahr-export","exec"),_boot)
_boot["install_notebook_package"](ROOT)

"""Notebook11 fixed epoch20 probes and six-seed registered primary evidence."""
from pathlib import Path
import copy, hashlib, json, math, sys, time
import numpy as np
import torch

ROOT=Path(__file__).resolve().parents[2]
if 'fmm11_runner' not in sys.modules:
    import types
    _path=ROOT/'experiments/11_manifold_long/runner.ipynb'
    _module=types.ModuleType('fmm11_runner');_module.__file__=str(_path);sys.modules['fmm11_runner']=_module
    for _cell in json.loads(_path.read_text())['cells']:
        if _cell['cell_type']=='code' and 'ahr-export' in _cell.get('metadata',{}).get('tags',[]):
            exec(compile(''.join(_cell['source']),str(_path),'exec'),_module.__dict__)
RUNNER=sys.modules['fmm11_runner']
_BASE=RUNNER.load_tagged(ROOT,'experiments/05_scaling/evaluation.ipynb','fmm11_inherited_evaluation')
for _name,_value in list(_BASE.__dict__.items()):
    if not _name.startswith('__') and _name not in globals():globals()[_name]=_value
build_model=RUNNER.build_model


def _evaluation_export_hash(root):
    return _digest([''.join(c['source']) for c in _read(Path(root)/'experiments/11_manifold_long/evaluation.ipynb')['cells']
        if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[])])

def _verify_plan_sources(root,plan):
    actual=RUNNER.stage_definition_hashes(root)
    if plan['source_hashes']!=actual:raise ValueError('Notebook11 registered definitions changed')
    if plan.get('consumer_notebook_sha256'):
        code=[''.join(c['source']) for c in _read(Path(root)/plan['resume_notebook'])['cells'] if c['cell_type']=='code']
        if _digest(code)!=plan['consumer_notebook_sha256']:raise ValueError('Registered main notebook code changed')
    return actual

def _stage_jobs(plan,training):
    if training.get('status')!='COMPLETE' or not training.get('equal_updates') or not training.get('paired_traces_within_seed') or not training.get('no_failed_attempts'):
        raise ValueError('Only clean, fully paired, equal-update training may enter quality evaluation')
    if set(training['arms'])!={a['id'] for a in plan['arms']}:raise ValueError('Training arm identities differ')
    jobs=[]
    for arm in plan['arms']:
        if arm['config']['training']['updates']!=plan['updates'] or arm['config']['probe']['epochs']!=20:
            raise ValueError('Schedule/probe budget differs')
        if arm['config']['evaluation'].get('heldout_enabled',False):raise ValueError('Official test remains reserved')
        record=training['arms'][arm['id']]
        if record['successful_updates']!=plan['updates'] or record['attempted_updates']!=plan['updates'] or record['failures']:
            raise ValueError('Incomplete or numerically failed arm')
        jobs.append({'model_id':arm['id']+'-initial','arm':arm,'updates':0,
            'control':'initialization','state':record['initial_teacher'],'training_cost':None})
        for milestone in plan['milestones']:
            state=record['milestones'][str(milestone)]
            if state['successful_updates']!=milestone:raise ValueError('Milestone update differs')
            jobs.append({'model_id':arm['id']+'-u'+str(milestone),'arm':arm,'updates':milestone,
                'control':'trained','state':state,'training_cost':state})
    return jobs

def _seed_statistics(rows,value_key='delta_percentage_points',expected_n=None):
    from scipy.stats import t, binomtest
    n=len(rows)
    if n<2 or len({int(row['seed']) for row in rows})!=n or (expected_n is not None and n!=int(expected_n)):
        raise ValueError('Distinct paired training seeds and the registered sample count are required')
    values=np.asarray([float(row[value_key]) for row in rows],dtype=np.float64)
    if not np.isfinite(values).all():raise ValueError('Nonfinite paired seed effect')
    mean=float(values.mean());sd=float(values.std(ddof=1));se=sd/math.sqrt(n);df=n-1
    half=float(t.ppf(.975,df))*se
    statistic=mean/se if se>0 else 0. if mean==0 else None
    p=float(2*t.sf(abs(statistic),df)) if statistic is not None else None
    positive=int((values>0).sum());negative=int((values<0).sum());ties=int((values==0).sum());nonzero=positive+negative
    sign_p=float(binomtest(positive,nonzero,.5,alternative='two-sided').pvalue) if nonzero else 1.
    return {'n_training_seeds':n,'degrees_of_freedom':df,'mean_percentage_points':mean,
        'sd_percentage_points':sd,'standard_error_percentage_points':se,
        'approximate_t_ci95_percentage_points':[mean-half,mean+half],
        't_statistic':statistic,'paired_t_two_sided_p_value':p,'exploratory_two_sided_p_value':p,
        'zero_variance_deltas':se==0,'range_percentage_points':[float(values.min()),float(values.max())],
        'positive_seeds':positive,'negative_seeds':negative,'zero_seeds':ties,
        'exact_two_sided_sign_test':{'p_value':sign_p,'positive_pairs':positive,'negative_pairs':negative,
            'tied_pairs_excluded':ties,'non_tied_pairs':nonzero,
            'null':'Positive and negative non-tied paired differences are equally likely; this checks direction consistency, not the mean-effect null.'},
        'paired_seed_deltas':[{'seed':int(row['seed']),'delta_percentage_points':float(row[value_key])} for row in rows],
        'uncertainty':'Paired training-seed t interval, conditional on reused development images. Six pairs improve variance estimation but do not guarantee significance; t inference assumes an appropriate paired-effect distribution. Exact sign test is a separate direction check. Secondary contrasts are exploratory and unadjusted.'}

def analyze_stage_evidence(evaluations):
    index={(value['variant'],int(value['seed']),int(value['updates'])):value for value in evaluations.values()}
    seeds=list(range(20261401,20261407));variants=['vit','spatial_kernel','learned_kernel']
    observed_steps=sorted({key[2] for key in index});final=max(observed_steps,default=0);steps=[0,350,final]
    if final not in (2400,1800,1200,1000) or observed_steps!=steps:
        raise ValueError('Expected initialization, milestone350 and a timing-only registered final update candidate')
    expected={(variant,seed,step)for variant in variants for seed in seeds for step in steps}
    if set(index)!=expected or len(index)!=len(evaluations):
        raise ValueError('All54 unique initial/milestone/final probes from six fresh threeway seed pairs are required')
    contrasts={};rows=[]
    def paired(name,values):
        contrasts[name]=_seed_statistics([{'seed':seed,'delta_percentage_points':value} for seed,value in zip(seeds,values)],expected_n=6)
    for variant in variants:
        for step in steps[1:]:
            paired(variant+'_learning_u'+str(step),[_paired_delta(index[(variant,seed,step)],index[(variant,seed,0)])for seed in seeds])
        paired(variant+'_continued_learning_u'+str(final),[_paired_delta(index[(variant,seed,final)],index[(variant,seed,350)])for seed in seeds])
    comparisons=[('spatial_kernel','vit'),('learned_kernel','vit'),('learned_kernel','spatial_kernel')]
    for candidate,control in comparisons:
        for step in steps:
            paired(candidate+'_minus_'+control+'_u'+str(step),[_paired_delta(index[(candidate,seed,step)],index[(control,seed,step)])for seed in seeds])
        for step in steps[1:]:
            paired(candidate+'_minus_'+control+'_learning_gain_u'+str(step),[
                _paired_delta(index[(candidate,seed,step)],index[(candidate,seed,0)])-_paired_delta(index[(control,seed,step)],index[(control,seed,0)])for seed in seeds])
    for variant in variants:
        for seed in seeds:
            initial=_score(index[(variant,seed,0)])
            for step in steps:
                saved=index[(variant,seed,step)];cost=saved.get('training_cost')or{};perf=cost.get('performance')or{}
                peak=perf.get('peak_cuda_allocated_bytes');diagnostic=saved['representation_diagnostics']['descriptive']
                rows.append({'model_id':saved['model_id'],'variant':variant,'kind':saved['kind'],'seed':seed,'updates':step,
                    'top1_percent':_score(saved),'initial_top1_percent':initial,'gain_percentage_points':_score(saved)-initial,
                    'image_exposures':int(cost.get('image_exposures',0)),
                    'cumulative_training_step_seconds':cost.get('cumulative_step_seconds')if step else 0.,
                    'median_training_step_seconds':(perf.get('complete_jepa_step_seconds')or{}).get('median'),
                    'peak_training_GiB':float(peak)/1024**3 if peak is not None else None,
                    'encoder_parameters':saved['parameter_counts']['context'],'trainable_parameters':saved['parameter_counts']['trainable'],
                    'inactive_context_scaffold_parameters':saved['parameter_counts'].get('inactive_context_scaffold',0),
                    'encoder_active_parameters':saved['parameter_counts']['context_active'],
                    'trainable_active_parameters':saved['parameter_counts']['trainable_active'],
                    'feature_std_mean':diagnostic['feature_std_mean'],'effective_rank':diagnostic['effective_rank']})
    sanity={variant:{str(seed):{'finite':bool(index[(variant,seed,final)]['representation_diagnostics']['finite_train_and_development']),
        'not_exactly_constant':bool(index[(variant,seed,final)]['representation_diagnostics']['development_representation_not_exactly_constant']and index[(variant,seed,final)]['representation_diagnostics']['training_representation_not_exactly_constant'])}for seed in seeds}for variant in variants}
    learning={variant:{'mean_gain_percentage_points':contrasts[variant+'_learning_u'+str(final)]['mean_percentage_points'],
        'interval_excludes_zero_positive':contrasts[variant+'_learning_u'+str(final)]['approximate_t_ci95_percentage_points'][0]>0,
        'interval_excludes_zero_negative':contrasts[variant+'_learning_u'+str(final)]['approximate_t_ci95_percentage_points'][1]<0}for variant in variants}
    gate={'passed':all(row['finite']and row['not_exactly_constant']for group in sanity.values()for row in group.values()),
        'status':'NUMERICAL_SANITY_ONLY','representation_checks':sanity,'own_learning':learning,
        'reason':'Finite nonconstant features are numerical checks; no automatic winner-based continuation.'}
    primary_key='learned_kernel_minus_vit_u'+str(final);primary_stats=contrasts[primary_key]
    primary={'registered':True,'contrast':primary_key,'endpoint':'Final fixed epoch 20  frozen-EMA development top1 difference in percentage points',
        'alpha':.05,'n_pairs':6,'mean_effect_statistics':primary_stats,
        'primary_mean_test':'Two-sided paired t test; assess the sign of the mean and95%interval separately from significance.',
        'direction_robustness_test':'Two-sided exact sign test excluding exactties; a separate directionalnull, not a second mean test.',
        'scope':'Future fresh training seeds conditional on reused development images; no heldout-test or general dataset claim. No superiority guarantee.'}
    assessment={'status':'REGISTERED_PRIMARY_RESULTS_COMPLETE','registered_pairs':6,'degrees_of_freedom':5,
        'confirmatory_primary_endpoint':primary_key,'primary_analysis':primary,
        'mechanism_comparisons':['learned_kernel_minus_spatial_kernel_u'+str(final),'spatial_kernel_minus_vit_u'+str(final)],
        'interpretation':'Report all six paired effects, primary pairedmean uncertainty and exactsign direction check. Secondary contrasts and milestonelearning gains remain exploratory; no automatic selectedwinner.'}
    return {'contrasts':contrasts,'chart_rows':rows,'learning_gate':gate,'hypothesis_assessment':assessment,'primary_analysis':primary}

def write_stage_figures(artifact,result,summary):
    """All six observed seed effects; mean and explicitly labelled t intervals."""
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    variants=['vit','spatial_kernel','learned_kernel'];labels=['ViT','Spatial kernel','Learned geometry kernel']
    colors=['#59636E','#4B77A8','#697C53'];markers=['o','s','^'];seeds=list(range(20261401,20261407))
    steps=sorted({row['updates']for row in result['chart_rows']});final=max(steps)
    synthetic=result.get('evidence_role')=='SYNTHETIC_SOFTWARE_VALIDATION'
    subtitle='SYNTHETIC_SOFTWARE_VALIDATION: invented fixture values, no experiment results'if synthetic else'Food101 reused development images; fixed epoch20 probes; six fresh paired training seeds'
    fig,axes=plt.subplots(1,2,figsize=(13,4.8),layout='constrained')
    offsets=np.linspace(-.17,.17,len(seeds))
    for i,variant in enumerate(variants):
        rows=[row for row in result['chart_rows']if row['variant']==variant]
        for seed in seeds:
            values=sorted([row for row in rows if row['seed']==seed],key=lambda row:row['updates'])
            axes[0].plot([row['updates']for row in values],[row['top1_percent']for row in values],color=colors[i],marker=markers[i],linewidth=.8,alpha=.35)
        means=[float(np.mean([row['top1_percent']for row in rows if row['updates']==step]))for step in steps]
        axes[0].plot(steps,means,color=colors[i],marker=markers[i],linewidth=2,label=labels[i]+' mean')
        effects=result['contrasts'][variant+'_learning_u'+str(final)]['paired_seed_deltas']
        for offset,row in zip(offsets,effects):axes[1].scatter(row['delta_percentage_points'],i+offset,color=colors[i],marker=markers[i],s=40)
        axes[1].scatter(result['contrasts'][variant+'_learning_u'+str(final)]['mean_percentage_points'],i,color='#222222',marker='|',s=110)
    axes[0].set(title='Each seed and mean: initialization, 350 and final',xlabel='Successful training updates',ylabel='Development top1 accuracy (%)',xticks=steps)
    axes[0].set_ylim(bottom=0);axes[0].legend(frameon=False,fontsize=9)
    axes[1].axvline(0,color='#59636E',linewidth=1);axes[1].set(title='Final learning over own initialization: six seeds and mean',xlabel='Gain (percentage points)',yticks=range(3),yticklabels=labels)
    for axis in axes:axis.spines[['top','right']].set_visible(False);axis.grid(axis='x',alpha=.15)
    fig.suptitle(subtitle,fontsize=11,color='#9E2727'if synthetic else'#222222')
    quality=Path(artifact)/'stage_quality.png';fig.savefig(quality,dpi=160);plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(12,4.6),layout='constrained')
    for axis,key,title,unit in ((axes[0],'step_seconds','Full JEPA training step time','Seconds'),(axes[1],'peak_GiB','Peak allocated GPU memory','GiB')):
        values=[row[key]for row in summary]
        if any(value is None for value in values):axis.text(.5,.5,'UNKNOWN: incomplete measurements',ha='center',transform=axis.transAxes)
        else:
            axis.bar(range(3),values,color=colors,edgecolor='#333333',linewidth=.5)
            for i,value in enumerate(values):axis.annotate(f'{value:.3f}',(i,value),xytext=(0,4),textcoords='offset points',ha='center')
            axis.set_ylim(0,max(values)*1.18)
        axis.set(title=title,ylabel=unit,xticks=range(3),xticklabels=['ViT','Spatial','Learned geometry']);axis.spines[['top','right']].set_visible(False)
    fig.suptitle(subtitle if synthetic else'Means of six per-seed measurements; equal exposure, different compute',fontsize=11,color='#9E2727'if synthetic else'#222222')
    compute=Path(artifact)/'stage_compute.png';fig.savefig(compute,dpi=160);plt.close(fig)
    comparisons=['learned_kernel_minus_spatial_kernel_u'+str(final),'learned_kernel_minus_vit_u'+str(final),'spatial_kernel_minus_vit_u'+str(final)]
    names=['Learned geometry minus spatial kernel (secondary)','Learned geometry kernel minus ViT (primary)','Spatial kernel minus ViT (secondary)']
    fig,axis=plt.subplots(figsize=(11,4.5),layout='constrained')
    for i,key in enumerate(comparisons):
        stats=result['contrasts'][key];mean=stats['mean_percentage_points'];lo,hi=stats['approximate_t_ci95_percentage_points']
        for offset,row in zip(offsets,stats['paired_seed_deltas']):axis.scatter(row['delta_percentage_points'],i+offset,color='#4B77A8',marker='o',s=40,zorder=3)
        axis.errorbar(mean,i,xerr=[[max(0.,mean-lo)],[max(0.,hi-mean)]],color='#222222',fmt='|',markersize=13,capsize=4,linewidth=1.3,zorder=4)
    axis.axvline(0,color='#59636E',linewidth=1);axis.set(yticks=range(3),yticklabels=names,xlabel='Final development accuracy difference (percentage points)',title='Six paired effects; mean and 95% training-seed t interval (df = 5)')
    axis.spines[['top','right']].set_visible(False);axis.grid(axis='x',alpha=.15)
    fig.suptitle(subtitle,fontsize=10,color='#9E2727'if synthetic else'#222222')
    mechanism=Path(artifact)/'stage_contrasts.png';fig.savefig(mechanism,dpi=160);plt.close(fig)
    records={name:{'path':str(path.resolve()),'sha256':digest_file(path)}for name,path in [('quality',quality),('compute',compute),('architecture_controls',mechanism)]}
    records['visual_inspection']='Synthetic fixture software validation only'if synthetic else'Pending inspection after real experiment completion.'
    atomic_json(Path(artifact)/'stage_figure_manifest.json',records)
    return records

def write_stage_report(artifact_dir,result):
    artifact=Path(artifact_dir);final=max(row['updates']for row in result['chart_rows']);rows=[row for row in result['chart_rows']if row['updates']==final]
    variants=['vit','spatial_kernel','learned_kernel'];seeds=list(range(20261401,20261407));summary=[]
    synthetic=result.get('evidence_role')=='SYNTHETIC_SOFTWARE_VALIDATION'
    for variant in variants:
        group=[row for row in rows if row['variant']==variant]
        def mean(name):return float(np.mean([row[name]for row in group]))if all(row[name]is not None for row in group)else None
        summary.append({'variant':variant,'top1_percent':mean('top1_percent'),'gain_percentage_points':mean('gain_percentage_points'),
            'step_seconds':mean('median_training_step_seconds'),'peak_GiB':mean('peak_training_GiB'),
            'encoder_parameters':mean('encoder_parameters'),'trainable_parameters':mean('trainable_parameters'),
            'inactive_context_scaffold_parameters':mean('inactive_context_scaffold_parameters'),
            'encoder_active_parameters':mean('encoder_active_parameters'),'trainable_active_parameters':mean('trainable_active_parameters'),
            'feature_std_mean':mean('feature_std_mean'),'effective_rank':mean('effective_rank')})
    indexed={row['variant']:row for row in summary};ratios={}
    for candidate,control in [('spatial_kernel','vit'),('learned_kernel','vit'),('learned_kernel','spatial_kernel')]:
        ratios[candidate+'_over_'+control]={}
        for field in ('step_seconds','peak_GiB'):
            a=indexed[candidate][field];b=indexed[control][field]
            ratios[candidate+'_over_'+control][field]=a/b if a is not None and b is not None and b>0 else None
    atomic_json(artifact/'stage_cost_comparisons.json',ratios)
    figures=write_stage_figures(artifact,result,summary)
    title='# SYNTHETIC_SOFTWARE_VALIDATION — invented fixture values, no experiment results'if synthetic else'# Notebook11 overnight manifold kernel JEPA study'
    status='Software fixtures only; these values are not observed experiment outcomes.'if synthetic else f'Status: COMPLETE. Six fresh seed pairs completed all three architectures at {final} updates and all 54 initialization/350/final probes at fixed epoch 20. Official test: NOT_RUN.'
    lines=[title,'',status,'',
        'The single registered primary contrast is learned geometry kernel minus ViT at the timing-selected final update. Paired mean uncertainty and the exact sign direction check are reported. All other comparisons, intermediate milestones, initialization gains and costs are secondary. Development membership is reused.',
        '', '| Variant | Final development top1 (%) | Gain over initialization (pp) | Median step (s) | Peak allocated memory (GiB) |',
        '|---|---:|---:|---:|---:|']
    for row in summary:
        def show(key):return 'UNKNOWN'if row[key]is None else f"{row[key]:.4f}"
        lines.append('| '+row['variant']+' | '+' | '.join(show(key)for key in('top1_percent','gain_percentage_points','step_seconds','peak_GiB'))+' |')
    primary=result['primary_analysis'];stats=primary['mean_effect_statistics'];lo,hi=stats['approximate_t_ci95_percentage_points']
    p=stats['paired_t_two_sided_p_value'];p_text='UNDEFINED (zero sample variance)'if p is None else f'{p:.6g}'
    sign=stats['exact_two_sided_sign_test']
    lines+=['',f"Primary final contrast: mean {stats['mean_percentage_points']:+.4f} pp; 95% paired t interval [{lo:.4f},{hi:.4f}] pp, df = 5; two-sided paired t p = {p_text}.",
        f"Direction robustness: exact two-sided sign-test p = {sign['p_value']:.6g}; {sign['positive_pairs']} positive, {sign['negative_pairs']} negative and {sign['tied_pairs_excluded']} tied pairs. Ties are excluded. This tests direction balance rather than the mean-effect null.",
        '', '| Fresh paired seed | Learned minus ViT final (pp) | Learned minus spatial final (pp) |',
        '|---|---:|---:|']
    primary_deltas={row['seed']:row['delta_percentage_points']for row in stats['paired_seed_deltas']}
    spatial=result['contrasts']['learned_kernel_minus_spatial_kernel_u'+str(final)]
    spatial_deltas={row['seed']:row['delta_percentage_points']for row in spatial['paired_seed_deltas']}
    for seed in seeds:lines.append(f'|{seed}|{primary_deltas[seed]:+.4f}|{spatial_deltas[seed]:+.4f}|')
    lines+=['','The common patch stem, every block MLP/norm1/norm2, final norm and predictor share initialization with independent storage. Spatial and learned kernels additionally share value/output projections, bandwidths and mixture weights. Learned geometry adds active coordinate maps and starts with different interactions; initialization controls are retained. This is an architecture comparison, not a parameter-matched proof about manifolds.',
        '', 'Kernel sums are evaluated densely. No FMM acceleration, asymptotic speedup, established data manifold or coordinate isomorphism is claimed. The unchanged JEPA predictor retains Transformer blocks. Equal exposure does not mean equal compute.',
        '', '| Secondary final/gain contrast | Mean (pp) | 95% paired t interval, df = 5 (pp) | Direction sign-test p |',
        '|---|---:|---:|---:|']
    for name,entry in result['contrasts'].items():
        if name.endswith('u'+str(final)) and name!=primary['contrast']:
            low,high=entry['approximate_t_ci95_percentage_points']
            lines.append(f"|{name}|{entry['mean_percentage_points']:+.4f}|[{low:.4f},{high:.4f}]|{entry['exact_two_sided_sign_test']['p_value']:.6g}|")
    lines+=['','| Variant | Encoder total | Encoder active | Trainable total | Trainable active | Mean feature std | Effective rank |',
        '|---|---:|---:|---:|---:|---:|---:|']
    for row in summary:lines.append(f"|{row['variant']}|{row['encoder_parameters']:.0f}|{row['encoder_active_parameters']:.0f}|{row['trainable_parameters']:.0f}|{row['trainable_active_parameters']:.0f}|{row['feature_std_mean']:.6f}|{row['effective_rank']:.3f}|")
    lines+=['','Training seeds are the replication units. Six pairs do not guarantee significance, and the t interval assumes a suitable paired-effect distribution. The sign test is a separate direction check. Secondary p values are exploratory and not multiplicity-adjusted. An interval containing zero proves neither equivalence nor absence of an effect. Reused development images limit the claim; the official test stays reserved.',
        '', 'Feature diagnostics use the same predeclared subset before probe standardization. Geometry diagnostics use the same first two development images at every endpoint and record their actual device. Finite nonconstant features and sampled kernel compressibility establish neither useful learning nor an FMM speedup.',
        '', 'Final updates were selected solely from discarded timing/memory calibration, from 2400/1800/1200/1000, before quality outcomes. All 18 arms retain identical optimizer schedules, exposure and milestones 350/final. The original twelve-hour clock includes recovery and cannot reset.',
        '', 'Step timing includes ordinary preprocessing and full JEPA training compute; it excludes probe/checkpoint overhead. Peak memory is allocated CUDA memory, excluding reserved cache and CUDA context. Immutable states, caches and reconstructed predictions support reported values.',
        '', 'Resume: Run All in Notebook11 with the same run ID and unchanged sources before the original deadline. An exhausted deadline never extends automatically.',
        '', '![Six seeds and mean before and after learning](stage_quality.png)',
        '', '![Primary and secondary paired accuracy differences](stage_contrasts.png)',
        '', '![Measured training time and allocated memory](stage_compute.png)',
        '', 'Descriptive cost ratios: '+json.dumps(ratios,sort_keys=True)+'. No analytic scaling or universal speed guarantee follows.']
    path=artifact/'stage_results.md';path.write_text('\n'.join(lines)+'\n');atomic_json(artifact/'stage_summary_table.json',summary)
    return {'report':str(path.resolve()),'summary_table':str((artifact/'stage_summary_table.json').resolve()),'figures':figures,'cost_comparisons':ratios}

def _evaluate_stage_locked(root, artifact, plan, bundle, training, deadline):
    sources = _verify_plan_sources(root, plan)
    membership = _fixed_membership(root, bundle)
    jobs = _stage_jobs(plan, training)
    stage_role = plan.get("stage_role", plan["stage_id"])
    if stage_role not in STAGE_ROLES: raise ValueError("Unknown stage_role")
    if membership["pretraining_count"] != (65600 if stage_role == "data_volume" else 10100):
        raise ValueError("Pretraining count differs from the registered stage role")
    if int(plan["arms"][0]["config"]["image_size"]) != (256 if stage_role == "resolution" else 128):
        raise ValueError("Stage image resolution differs from declared role")
    identity = {"schema": "notebook11-stage-evaluation/v1", "plan": _canonical(plan),
        "source_hashes": sources, "evaluation_export_hash": _evaluation_export_hash(root),
        "dataset_fingerprint": bundle["dataset"].fingerprint, "membership": membership,
        "jobs": _canonical(jobs), "probe_selection": "fixed_final_epoch20", "official_test": "NOT_RUN"}
    identity_hash = _digest(identity)
    manifest_path = artifact/"stage_evaluation_manifest.json"
    if manifest_path.exists():
        manifest = _read(manifest_path)
        if manifest["identity_hash"] != identity_hash: raise ValueError("Stage evaluation identity changed; refusing stale reuse")
    else:
        manifest = {"identity_hash": identity_hash, "identity": identity, "completed": {}, "started_unix": time.time()}
        atomic_json(manifest_path, manifest)
    evaluations = {}
    for job in jobs:
        model_id, arm = job["model_id"], job["arm"]
        config = copy.deepcopy(arm["config"])
        summary_path = artifact/"evaluation"/(model_id+".json")
        _verified_record(job["state"])
        if model_id in manifest["completed"]:
            evaluations[model_id] = _load_saved_job(summary_path, manifest["completed"][model_id], identity_hash, job, membership)
            continue
        _deadline_check(deadline, 5)
        configure_environment(config)
        atomic_json(artifact/"evaluation_status.json", {"status":"RUNNING", "model_id":model_id,
            "completed_models":len(evaluations), "total_models":len(jobs), "updated_unix":time.time()})
        model = encoder = None
        try:
            _release_semantic(config["device"])
            model = build_model(config, arm["kind"]).cpu()
            parameter_counts={
                'context':sum(p.numel() for p in model.context.parameters()),
                'predictor':sum(p.numel() for p in model.predictor.parameters()),
                'teacher':sum(p.numel() for p in model.target.parameters()),
                'trainable':sum(p.numel() for p in model.parameters() if p.requires_grad),
                'inactive_context_scaffold':0}
            parameter_counts['context_active']=parameter_counts['context']-parameter_counts['inactive_context_scaffold']
            parameter_counts['trainable_active']=parameter_counts['trainable']-parameter_counts['inactive_context_scaffold']
            state = torch.load(job["state"]["path"], map_location="cpu", weights_only=job["updates"] == 0)
            if job["updates"] == 0:
                model.target.load_state_dict(state)
            else:
                if int(state["state"]["successful_updates"]) != job["updates"] or state["config"] != arm["config"]:
                    raise ValueError("Checkpoint milestone/config differs from the registered schedule")
                model.load_state_dict(state["model"])
            del state
            encoder = model.target
            model = None
            common = {"dataset_revision":bundle["dataset"].fingerprint, "checkpoint_hash":encoder_state_hash(encoder),
                "encoder_kind":getattr(encoder,"kind",arm["kind"]), "preprocessing":preprocessing_descriptor(config),
                "pooling":"mean_final_layernorm_patches",
                "normalization":"encoder final affine LayerNorm then patch mean; train-only per-feature probe standardization"}
            caches, extraction = {}, []
            for split, ids in (("train", membership["probe_train_ids"]), ("development", membership["development_ids"])):
                _deadline_check(deadline, 2)
                dataset = _DeadlineDataset(EvaluationDataset(bundle["dataset"], ids, config), deadline)
                print(stage_role, model_id, split, len(ids), "fixed labeled feature IDs", flush=True)
                cache = extract_features(encoder, dataset, artifact/"features"/model_id,
                    {**common,"sample_ids":ids}, batch_size=int(config["evaluation"]["batch_size"]), device=config["device"])
                caches[split] = cache
                extraction.append({"split":split, "cache_reused":cache["reused"], **cache["performance"]})
            geometry_ids=list(membership["development_ids"][:2])
            geometry={"status":"NOT_APPLICABLE","reason":"ViT has no declared kernel geometry",
                "sample_ids":geometry_ids,"scope":"Predeclared first two development images; descriptive only, no model/probe selection."}
            if arm["variant"] in ("spatial_kernel","learned_kernel"):
                geometry_dataset=EvaluationDataset(bundle["dataset"],geometry_ids,config)
                geometry_images=torch.stack([geometry_dataset[i]["image"] for i in range(len(geometry_ids))]).to(next(encoder.parameters()).device)
                backend=RUNNER.load_tagged(root,"experiments/11_manifold_long/encoder.ipynb","fmm11_encoder")
                with torch.no_grad():geometry=backend.kernel_geometry_diagnostics(encoder,geometry_images,max_samples=2)
                geometry.update(sample_ids=geometry_ids,diagnostic_device=str(next(encoder.parameters()).device),scope="Predeclared first two development images; descriptive only, no model/probe selection.")
                del geometry_images,geometry_dataset
            encoder.cpu()
            _release_semantic(config["device"])
            diagnostics = _cached_representation_diagnostics(caches["train"], caches["development"], bundle.get("diagnostic_ids", []))
            result = _fit_fixed_probe(caches["train"], caches["development"],
                root/"checkpoints"/plan["run_id"]/"stages"/plan["stage_id"]/"probes", config, model_id, encoder, deadline, plan)
            saved = {"identity_hash":identity_hash, "model_id":model_id, "arm_id":arm["id"], "kind":arm["kind"], "variant":arm["variant"],
                "seed":int(arm["seed"]), "stage_role":stage_role, "updates":job["updates"], "control":job["control"],
                "source_encoder":job["state"], "feature_caches":{name:_state_record(cache["path"]) for name,cache in caches.items()},
                "probe":result, "parameter_counts":parameter_counts, "representation_diagnostics":diagnostics, "kernel_geometry_diagnostics":geometry, "feature_extraction":extraction,
                "training_cost":job["training_cost"], "heldout_status":"NOT_RUN"}
            write_csv(artifact/"predictions"/(model_id+".csv"), result["predictions"])
            atomic_json(summary_path, saved)
            manifest["completed"][model_id] = _state_record(summary_path)
            atomic_json(manifest_path, manifest)
            evaluations[model_id] = saved
            print(stage_role, model_id, "fixed epoch20 top1", 100*result["summary"]["metrics"]["top1"], flush=True)
        finally:
            model = encoder = None
            _release_semantic(config["device"])
    _deadline_check(deadline)
    if len(evaluations) != len(jobs): raise AssertionError("Every registered initial/350/final probe is required")
    probe_inits = {x["probe"]["summary"]["initialization_hash"] for x in evaluations.values()}
    probe_budgets = {_digest(x["probe"]["summary"]["budget"]) for x in evaluations.values()}
    probe_orders = {_digest([(r["epoch"],r["split"],r["permutation_hash"]) for r in x["probe"]["metrics"]]) for x in evaluations.values()}
    if len(probe_inits) != 1 or len(probe_budgets) != 1 or len(probe_orders) != 1:
        raise AssertionError("Probe initialization/budget/order is not identical across stage comparisons")
    scientific = analyze_stage_evidence(evaluations)
    result = {"schema":"notebook11-stage-evaluation/v1", "status":"COMPLETE", "run_id":plan["run_id"],
        "stage_id":plan["stage_id"], "stage_role":stage_role, "identity_hash":identity_hash,
        "plan_hash":_digest(plan), "dataset_fingerprint":bundle["dataset"].fingerprint,
        "membership":membership, "evaluations":evaluations, "milestones":[0]+list(plan["milestones"]),
        "schedule_note":"Six fresh paired seeds and checkpoints 350/final fixed; final updates selected only from discarded timing/memory calibration before quality outcomes. All 0/350/final encoders use fixed epoch 20 probes.",
        "image_size":int(plan["arms"][0]["config"]["image_size"]), "pretraining_images":len(bundle["train_ids"]),
        "training_design":{"updates":int(plan["updates"]),"effective_batch":plan["arms"][0]["config"]["training"]["microbatch"]*plan["arms"][0]["config"]["training"]["accumulation_steps"],
            "lr":plan["arms"][0]["config"]["training"]["lr"],"ema_start":.99,
            "weight_decay":plan["arms"][0]["config"]["training"]["weight_decay"],
            "warmup_fraction":plan["arms"][0]["config"]["training"]["warmup_fraction"],
            "patch_size":plan["arms"][0]["config"]["model"]["patch_size"],
            "model":copy.deepcopy(plan["arms"][0]["config"]["model"]), "dtype":plan["arms"][0]["config"].get("dtype","float32") },
        "probe_selection":"fixed_final_epoch20", "labels_in_self_supervised_loss":False,
        "official_test":"NOT_RUN", "started_unix":manifest["started_unix"], "completed_unix":time.time(),
        "learning_gate":scientific["learning_gate"], "hypothesis_assessment":scientific["hypothesis_assessment"], "primary_analysis":scientific["primary_analysis"], "contrasts":scientific["contrasts"], "chart_rows":scientific["chart_rows"],
        "report_scope":"Six fresh paired seeds, threearchitectures, milestone350/final and54 probes. Single preregistered learned minus ViT primary contrast; secondary contrasts exploratory; development images reused."}
    atomic_json(artifact/"stage_evaluation.json", result)
    result["presentation"] = write_stage_report(artifact, result)
    atomic_json(artifact/"stage_evaluation.json", result)
    atomic_json(artifact/"evaluation_status.json", {"status":"COMPLETE", "completed_models":len(jobs), "learning_gate":result["learning_gate"], "updated_unix":time.time()})
    return result

def verify_evaluation_smoke(root,artifact,plan,bundle,deadline_unix):
    """Validate real encoder/cache integration and the unchanged fixed20epoch probe path.

    Old07features are used only as validation inputs, never as11quality evidence.
    Fresh11teacher features use two real training and two real development images per architecture.
    """
    root,artifact=Path(root).resolve(),Path(artifact).resolve();artifact.relative_to(root)
    artifact.mkdir(parents=True,exist_ok=True)
    old_root=root/'artifacts/notebook_07/energy-encoder-001'
    old_summary=old_root/'evaluation_recovery_v1/evaluation/s20261011-energy-initial.json'
    old_plan_path=old_root/'stages/learning/plan.json'
    source_summary=_read(old_summary);old_plan=_read(old_plan_path)
    provenance={'summary':_state_record(old_summary),'plan':_state_record(old_plan_path),
        'teacher':source_summary['source_encoder'],'caches':source_summary['feature_caches']}
    identity={'schema':'notebook11-evaluation-smoke/v1','plan_hash':_digest(plan),
        'source_hashes':RUNNER.stage_definition_hashes(root),'validation_only_source07':provenance,
        'fresh_training_pixels':list(bundle['train_ids'][:2]),'fresh_development_pixels':list(bundle['dev_ids'][:2]),
        'scope':'Threefreshencoderkind realfeature/tinyprobeintegration plus interrupted-resumed fullfinalepoch20probe on old07validationcaches; excludedfrom11scientificresults.'}
    record_path=artifact/'evaluation_smoke.json'
    if record_path.exists():
        record=_read(record_path)
        if record['identity']!=identity or record.get('status')!='PASS':raise ValueError('Evaluation smoke identity/status changed')
        for record_file in record['verified_outputs']:_verified_record(record_file)
        return record
    _deadline_check(deadline_unix,5)
    outputs=[];checks={}
    for variant in plan['variants']:
        arm=next(a for a in plan['arms'] if a['variant']==variant)
        config=copy.deepcopy(arm['config']);configure_environment(config)
        models,initial=RUNNER.initialize_models(config,artifact/'fresh_initial_models'/variant)
        model=models[arm['kind']];del models
        encoder=model.target
        caches={}
        for split,ids in [('train',list(bundle['train_ids'][:2])),('development',list(bundle['dev_ids'][:2]))]:
            dataset=_DeadlineDataset(EvaluationDataset(bundle['dataset'],ids,config),deadline_unix)
            feature_identity={'dataset_revision':bundle['dataset'].fingerprint,'checkpoint_hash':encoder_state_hash(encoder),
                'encoder_kind':encoder.kind,'preprocessing':preprocessing_descriptor(config),
                'pooling':'mean_final_layernorm_patches',
                'normalization':'encoder final affine LayerNorm then patch mean; train-only per-feature probe standardization','sample_ids':ids}
            cache=extract_features(encoder,dataset,artifact/'fresh_initial_features'/variant,feature_identity,batch_size=2,device=config['device'])
            if cache['identity']['encoder_kind']!=variant or cache['features'].shape!=(2,192):raise AssertionError('Fresh encoder integration mismatch')
            caches[split]=cache;outputs.append(_state_record(cache['path']))
        tiny_config=copy.deepcopy(config);tiny_config['probe'].update(epochs=20,batch_size=256)
        tiny_root=root/'checkpoints'/plan['run_id']/'validation'/'tiny_actual_kind_probes'/variant
        with _probe_checkpoint_metadata(plan):
            reference=fit_linear_probe(caches['train'],caches['development'],tiny_root,tiny_config,'reference',encoder=encoder)
            partial=fit_linear_probe(caches['train'],caches['development'],tiny_root,tiny_config,'resumed',encoder=encoder,stop_after_updates=1)
            resumed=fit_linear_probe(caches['train'],caches['development'],tiny_root,tiny_config,'resumed',encoder=encoder)
        reference_state=_load_probe_checkpoint(tiny_root/'reference',reference['selected_probe']['identity'])
        resumed_state=_load_probe_checkpoint(tiny_root/'resumed',resumed['selected_probe']['identity'])
        if reference_state['successful_updates']!=20 or resumed_state['successful_updates']!=20 or resumed_state['completed_epochs']!=20:
            raise AssertionError('Tiny real-kind epoch20 probe did not finish')
        for key,tensor in reference_state['classifier'].items():
            if not torch.equal(tensor,resumed_state['classifier'][key]):raise AssertionError('Actual-kind probe resume changed the final classifier')
        x=standardize(caches['development']['features'],resumed_state['standardization'])
        reconstructed=F.linear(x,resumed_state['classifier']['weight'],resumed_state['classifier']['bias'])
        if not torch.isfinite(reconstructed).all():raise AssertionError('Nonfinite tiny real-kind probe reconstruction')
        for name in ('reference','resumed'):outputs.append(_state_record(tiny_root/name/'probe_manifest.json'))
        checks[variant]={'status':'PASS','actual_encoder_kind':encoder.kind,'train_examples':2,'development_examples':2,
            'tiny_probe_completed_epochs':20,'tiny_probe_updates':20,'interrupted_resumed_classifier_bitwise_equal':True,
            'final_logits_reconstructed':True,'quality_result_use':'FORBIDDEN_validation_only'}
        model=encoder=cache=caches=reference=resumed=reference_state=resumed_state=None;_release_semantic(config['device'])
    _deadline_check(deadline_unix,5)
    old_arm=next(a for a in old_plan['arms'] if a['id']==source_summary['arm_id'])
    probe_config=copy.deepcopy(old_arm['config']);configure_environment(probe_config)
    with RUNNER.isolated_rng(int(old_arm['seed'])):
        fixture_runner=RUNNER.load_tagged(root,'experiments/07_energy/runner.ipynb','fmm11_probe_fixture_runner')
        model=fixture_runner.build_model(probe_config,old_arm['kind']).cpu()
    model.target.load_state_dict(torch.load(_verified_record(source_summary['source_encoder']),map_location='cpu',weights_only=True))
    encoder=model.target;caches={}
    for split,record in source_summary['feature_caches'].items():
        path=_verified_record(record);metadata=_read(path.with_suffix('.json'))
        if metadata['sha256']!=digest_file(path):raise ValueError('Validation-only07cache metadata mismatch')
        cache=torch.load(path,map_location='cpu',weights_only=True);validate_feature_cache(cache,metadata['identity'])
        if cache['identity']['encoder_kind']!='energy' or cache['identity']['checkpoint_hash']!=encoder_state_hash(encoder):
            raise ValueError('Validation-only07energy encoder/cache identity mismatch')
        caches[split]=cache
    checkpoint_root=root/'checkpoints'/plan['run_id']/'validation'/'evaluation_smoke_probes'
    model_id='validation-only-07-energy-initial'
    with _probe_checkpoint_metadata(plan):
        partial=fit_linear_probe(caches['train'],caches['development'],checkpoint_root,probe_config,model_id,encoder=encoder,stop_after_updates=17)
    if partial.get('status')!='INTERRUPTED' and not (checkpoint_root/model_id/'probe_manifest.json').exists():
        raise AssertionError('Probe lifecycle failed before resumable checkpoint')
    fixed=_fit_fixed_probe(caches['train'],caches['development'],checkpoint_root,probe_config,model_id,encoder,deadline_unix,plan)
    if fixed['reconciliation']['status']!='PASS' or fixed['summary']['selected_epoch']!=20 or fixed['summary']['successful_probe_updates']!=800:
        raise AssertionError('Fixed epoch20 probe did not reconstruct')
    for metric in ('top1','top5','loss'):
        if not math.isclose(fixed['summary']['metrics'][metric],source_summary['probe']['summary']['metrics'][metric],rel_tol=1e-6,abs_tol=1e-7):
            raise AssertionError('Interrupted/resumed validation probe differs from original fixed-probe result: '+metric)
    outputs.append(fixed['final_probe']);checks['fixed_probe']={'status':'PASS','selected_epoch':20,'successful_updates':800,
        'controlled_interruption_at':17,'predictions_reconstructed':fixed['reconciliation']['predictions_reconstructed'],
        'matched_original07_fixed_probe':True,'quality_result_use':'FORBIDDEN_validation_only'}
    model=encoder=None;_release_semantic(probe_config['device'])
    record={'status':'PASS','identity':identity,'checks':checks,'verified_outputs':outputs,'completed_unix':time.time()}
    atomic_json(record_path,record)
    return record

for _name in ('build_model','_evaluation_export_hash','_verify_plan_sources','_stage_jobs','_seed_statistics',
              'analyze_stage_evidence','write_stage_report','_evaluate_stage_locked'):
    setattr(_BASE,_name,globals()[_name])
_BASE.RESUME_NOTEBOOK=RUNNER.RESUME_NOTEBOOK
_BASE.RESUME_CELL=RUNNER.RESUME_CELL
evaluate_stage=_BASE.evaluate_stage
